# Stock Inventory Level Forecasting (No Unity Catalog)

End-to-end ML pipeline for forecasting daily stock inventory levels across multiple SKUs.

| Step | Description |
|------|-------------|
| 1 | **Configuration** — Set experiment directory and model names |
| 2 | **Synthetic Data** — Generate daily inventory for 5 SKUs with trend, seasonality & replenishment |
| 3 | **EDA** — Visualise time series patterns, distribution and seasonal effects |
| 4 | **Databricks AutoML** — Auto-train & tune forecasting models (10-minute budget) |
| 5 | **Prediction** — Load best model, generate 30-day forward forecasts |

In [0]:
# ── Configuration ─────────────────────────────────────────────────────────────────────
# Set your own path
EXPERIMENT_DIR = "/Users/mufajjul.ali@microsoft.com/experiments/inventory_forecast"

print(f"MLflow experiment dir : {EXPERIMENT_DIR}")

In [0]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

products   = [f"SKU-{i:03d}" for i in range(1, 6)]
start_date = datetime(2022, 1, 1)
end_date   = datetime(2024, 6, 30)
date_range = pd.date_range(start_date, end_date, freq="D")

records = []
for product_id in products:
    base_level    = np.random.randint(800, 1500)
    trend_slope   = np.random.uniform(-0.05, 0.15)
    seasonal_amp  = np.random.uniform(100, 300)
    weekly_phase  = np.random.uniform(0, 2 * np.pi)

    for i, date in enumerate(date_range):
        trend         = trend_slope * i
        weekly        = seasonal_amp * 0.3 * np.sin(2 * np.pi * i / 7 + weekly_phase)
        annual        = seasonal_amp * np.sin(2 * np.pi * i / 365.25)
        noise         = np.random.normal(0, 25)
        replenishment = 450 if (i % 28 == 0) else 0          # monthly restock spike
        inventory_level = max(0.0, base_level + trend + weekly + annual + noise + replenishment)
        records.append({
            "date":            date,
            "product_id":      product_id,
            "inventory_level": round(inventory_level, 2),
        })

inventory_df = pd.DataFrame(records)
print(f"Shape       : {inventory_df.shape}")
print(f"Date range  : {inventory_df['date'].min().date()} -> {inventory_df['date'].max().date()}")
print(f"Products    : {inventory_df['product_id'].unique().tolist()}")
print(f"Avg level   : {inventory_df['inventory_level'].mean():.1f}")
display(inventory_df.head(10))

In [0]:
# Group inventory_df by product_id (SKU) and aggregate inventory_level
sku_grouped = inventory_df.groupby("product_id")["inventory_level"].agg(["mean", "sum", "min", "max"]).reset_index()
display(sku_grouped)

In [0]:
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

inventory_df["day_of_week"] = inventory_df["date"].dt.dayofweek
inventory_df["month"]       = inventory_df["date"].dt.month

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle("Stock Inventory — Exploratory Data Analysis", fontsize=13, fontweight="bold")

# Time series per SKU
for pid in inventory_df["product_id"].unique():
    sub = inventory_df[inventory_df["product_id"] == pid].sort_values("date")
    axes[0, 0].plot(sub["date"], sub["inventory_level"], label=pid, alpha=0.75, linewidth=0.7)
axes[0, 0].set_title("Inventory Level Over Time (by SKU)")
axes[0, 0].set_xlabel("Date"); axes[0, 0].set_ylabel("Inventory Level")
axes[0, 0].legend(fontsize=8)
axes[0, 0].xaxis.set_major_formatter(mdates.DateFormatter("%Y-%m"))
plt.setp(axes[0, 0].xaxis.get_majorticklabels(), rotation=45)

# Distribution
axes[0, 1].hist(inventory_df["inventory_level"], bins=60, color="steelblue", edgecolor="white", alpha=0.85)
axes[0, 1].set_title("Distribution of Inventory Levels")
axes[0, 1].set_xlabel("Inventory Level"); axes[0, 1].set_ylabel("Frequency")

# Day-of-week pattern
dow_avg   = inventory_df.groupby("day_of_week")["inventory_level"].mean()
day_names = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]
axes[1, 0].bar(day_names, dow_avg.values, color="coral", edgecolor="white")
axes[1, 0].set_title("Avg Inventory by Day of Week")
axes[1, 0].set_xlabel("Day"); axes[1, 0].set_ylabel("Avg Inventory Level")

# Monthly seasonality
monthly_avg = inventory_df.groupby("month")["inventory_level"].mean()
axes[1, 1].plot(monthly_avg.index, monthly_avg.values, marker="o", color="seagreen", linewidth=1.8)
axes[1, 1].set_xticks(range(1, 13))
axes[1, 1].set_xticklabels(["Jan","Feb","Mar","Apr","May","Jun",
                             "Jul","Aug","Sep","Oct","Nov","Dec"], fontsize=8)
axes[1, 1].set_title("Monthly Average Inventory Level")
axes[1, 1].set_xlabel("Month"); axes[1, 1].set_ylabel("Avg Inventory Level")

plt.tight_layout()
plt.show()

print("\nSummary statistics:")
print(inventory_df["inventory_level"].describe().round(2))

In [0]:
# AutoML requires a Spark DataFrame (or a registered Delta table)
spark_df = spark.createDataFrame(
    inventory_df[["date", "product_id", "inventory_level"]]
)

spark_df.printSchema()
print(f"Total rows : {spark_df.count():,}")
display(spark_df.limit(10))

In [0]:
from databricks import automl
import mlflow

print("=" * 62)
print(" Databricks AutoML — Time Series Forecasting")
print("=" * 62)
print(f"  Target col    : inventory_level")
print(f"  Time col      : date")
print(f"  Identity cols : product_id  (one series per SKU)")
print(f"  Frequency     : daily")
print(f"  Horizon       : 30 days")
print(f"  Timeout       : 10 minutes")
print("=" * 62 + "\n")

automl_summary = automl.forecast(
    dataset=spark_df,
    target_col="inventory_level",
    time_col="date",
    identity_col=["product_id"],
    frequency="d",
    horizon=30,
    timeout_minutes=10,
    experiment_dir=EXPERIMENT_DIR,
)

best = automl_summary.best_trial
print("\nAutoML complete!")
print(f"  Experiment        : {automl_summary.experiment.name}")
print(f"  Best run ID       : {best.mlflow_run_id}")
print(f"  Best metrics      : {best.metrics}")
print(f"  Explore notebook  : {best.notebook_url}")

In [0]:
import mlflow
import matplotlib.pyplot as plt
import pandas as pd
from datetime import timedelta

best_run_id = automl_summary.best_trial.mlflow_run_id
model_uri   = f"runs:/{best_run_id}/model"
print(f"Loading model: {model_uri}\n")

loaded_model = mlflow.pyfunc.load_model(model_uri)

# Build future DataFrame — next 30 days x all 5 products
last_date      = inventory_df["date"].max()
future_records = [
    {"date": pd.Timestamp(d), "product_id": pid}
    for pid in inventory_df["product_id"].unique()
    for d in pd.date_range(last_date + timedelta(days=1), periods=30, freq="D")
]
future_df = pd.DataFrame(future_records)

print(f"Running inference for {len(future_df)} records ({inventory_df['product_id'].nunique()} SKUs x 30 days) ...")
predictions_raw = loaded_model.predict(future_df)

# Handle case where model returns a Series instead of DataFrame
if isinstance(predictions_raw, pd.Series):
    predictions_df = future_df.copy()
    predictions_df["yhat"] = predictions_raw.values
else:
    predictions_df = predictions_raw

print(f"Prediction columns : {predictions_df.columns.tolist()}")
display(predictions_df.head(15))

# ── Forecast visualisation ───────────────────────────────────────────────────────────────────
try:
    fig, ax = plt.subplots(figsize=(14, 5))
    for idx, pid in enumerate(inventory_df["product_id"].unique()):
        clr  = plt.cm.tab10.colors[idx % 10]
        hist = inventory_df[inventory_df["product_id"] == pid].tail(90).sort_values("date")
        ax.plot(hist["date"], hist["inventory_level"],
                color=clr, lw=0.9, alpha=0.6, label=f"{pid} historical")

        preds = (
            predictions_df[predictions_df["product_id"] == pid]
            if "product_id" in predictions_df.columns
            else predictions_df
        )
        t_col = next((col for col in ["ds", "date"] if col in preds.columns), None)
        v_col = next((col for col in ["yhat", "forecast", "inventory_level"]
                      if col in preds.columns), None)
        if t_col and v_col and not preds.empty:
            ax.plot(preds[t_col], preds[v_col], "--",
                    color=clr, lw=1.3, label=f"{pid} forecast")

    ax.set_title("30-Day Inventory Forecast (dashed) vs Historical (solid)")
    ax.set_xlabel("Date"); ax.set_ylabel("Inventory Level")
    ax.legend(fontsize=7, ncol=2)
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Forecast plot unavailable ({e}). Raw predictions above.")

In [0]:
import mlflow
from mlflow.tracking import MlflowClient
import cmdstanpy

# Use workspace model registry (not Unity Catalog)
mlflow.set_registry_uri("databricks")
MODEL_NAME = "inventory_forecast_model"

best_run_id = automl_summary.best_trial.mlflow_run_id
original_model_uri = f"runs:/{best_run_id}/model"

print(f"Best run ID: {best_run_id}")
print(f"Original model URI: {original_model_uri}\n")

# Fix: Re-log model with cmdstanpy pinned in conda env
# The serving environment needs cmdstanpy to deserialize Prophet models
fixed_conda_env = {
    "channels": ["conda-forge"],
    "dependencies": [
        "python=3.12.3",
        "pip<=25.0.1",
        {
            "pip": [
                "mlflow==3.0.1",
                "prophet==1.1.6",
                f"cmdstanpy=={cmdstanpy.__version__}",
                "cloudpickle==3.0.0",
                "databricks-automl-runtime==0.2.21",
            ]
        },
    ],
    "name": "mlflow-env",
}

print("Re-logging model with fixed conda environment (adding cmdstanpy)...")

import pandas as pd

# Load the model and unwrap the underlying PythonModel
loaded_model = mlflow.pyfunc.load_model(original_model_uri)
python_model = loaded_model.unwrap_python_model()
print(f"  Unwrapped model type: {type(python_model).__name__}")

# Get the original model's signature
original_model_info = mlflow.models.get_model_info(original_model_uri)
original_signature = original_model_info.signature
print(f"  Original signature: {original_signature}")


# Wrapper to fix date type coercion before prediction
class FixedProphetModel(mlflow.pyfunc.PythonModel):
    """Wraps the AutoML Prophet model to ensure date column is datetime64."""
    def __init__(self, inner_model):
        self.inner_model = inner_model

    def predict(self, context, model_input, params=None):
        # Convert date column to datetime64 to prevent merge type mismatch
        if "date" in model_input.columns:
            model_input["date"] = pd.to_datetime(model_input["date"])
        return self.inner_model.predict(context, model_input, params)


wrapped_model = FixedProphetModel(python_model)

# Re-log with fixed conda env + date coercion wrapper + signature
with mlflow.start_run() as run:
    mlflow.pyfunc.log_model(
        artifact_path="model",
        python_model=wrapped_model,
        conda_env=fixed_conda_env,
        signature=original_signature,
    )
    fixed_run_id = run.info.run_id

fixed_model_uri = f"runs:/{fixed_run_id}/model"
print(f"  Fixed run ID: {fixed_run_id}")
print(f"  Fixed model URI: {fixed_model_uri}\n")

# Register the fixed model
print(f"Registering to: {MODEL_NAME}")
result = mlflow.register_model(fixed_model_uri, MODEL_NAME)

print(f"  Name    : {result.name}")
print(f"  Version : {result.version}")
print(f"  Status  : {result.status}")

# Transition to Production stage
client = MlflowClient()
client.transition_model_version_stage(
    name=MODEL_NAME,
    version=result.version,
    stage="Production",
)
print(f"  Stage -> Production")

In [0]:
import requests
import json
import time

# Endpoint configuration
ENDPOINT_NAME = "inventory-forecast-endpoint"
MODEL_VERSION = result.version

# Get workspace URL and token
notebook_context = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
workspace_url = spark.conf.get("spark.databricks.workspaceUrl")
token = notebook_context.apiToken().get()

base_url = f"https://{workspace_url}"
headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}

# Define serving endpoint config
endpoint_config = {
    "name": ENDPOINT_NAME,
    "config": {
        "served_models": [
            {
                "model_name": MODEL_NAME,
                "model_version": MODEL_VERSION,
                "workload_size": "Small",
                "scale_to_zero_enabled": True,
            }
        ],
        "traffic_config": {
            "routes": [
                {
                    "served_model_name": f"{MODEL_NAME}-{MODEL_VERSION}",
                    "traffic_percentage": 100,
                }
            ]
        },
    },
}

print(f"Deploying endpoint: {ENDPOINT_NAME}")
print(f"  Model   : {MODEL_NAME} v{MODEL_VERSION}")
print(f"  Size    : Small (scale-to-zero enabled)\n")

# Create or update endpoint
resp = requests.get(f"{base_url}/api/2.0/serving-endpoints/{ENDPOINT_NAME}", headers=headers)
if resp.status_code == 200:
    print("Endpoint exists — updating...")
    resp = requests.put(
        f"{base_url}/api/2.0/serving-endpoints/{ENDPOINT_NAME}/config",
        headers=headers,
        json=endpoint_config["config"],
    )
else:
    print("Creating new endpoint...")
    resp = requests.post(
        f"{base_url}/api/2.0/serving-endpoints",
        headers=headers,
        json=endpoint_config,
    )

if resp.status_code in (200, 201):
    print(f"\nEndpoint deployed successfully!")
    print(f"  URL: {base_url}/serving-endpoints/{ENDPOINT_NAME}/invocations")
    print(f"\nNote: It may take a few minutes for the endpoint to become ready.")
    
    # Poll for readiness (up to 5 minutes)
    print("\nWaiting for endpoint to be ready...")
    for i in range(30):
        status_resp = requests.get(
            f"{base_url}/api/2.0/serving-endpoints/{ENDPOINT_NAME}",
            headers=headers,
        )
        state = status_resp.json().get("state", {}).get("ready", "UNKNOWN")
        if state == "READY":
            print(f"  Endpoint is READY!")
            break
        print(f"  [{i+1}/30] Status: {state} — waiting 10s...")
        time.sleep(10)
    else:
        print("  Timed out waiting. Check endpoint status in the UI.")
else:
    print(f"Error ({resp.status_code}): {resp.text}")

In [0]:
import requests
import json
import pandas as pd
from datetime import timedelta

# Build a sample payload — 5 days for SKU-001
last_date = inventory_df["date"].max()
sample_records = [
    {"date": str(pd.Timestamp(d).date()), "product_id": "SKU-001"}
    for d in pd.date_range(last_date + timedelta(days=1), periods=5, freq="D")
]

payload = {"dataframe_records": sample_records}

ENDPOINT_NAME = "inventory-forecast-endpoint"
print(f"Testing endpoint: {ENDPOINT_NAME}")

print ('full endpoint: ',f"{base_url}/serving-endpoints/{ENDPOINT_NAME}/invocations")
print(f"Payload ({len(sample_records)} records):")
print(json.dumps(payload, indent=2))
print()

resp = requests.post(
    f"{base_url}/serving-endpoints/{ENDPOINT_NAME}/invocations",
    headers=headers,
    json=payload,
)

if resp.status_code == 200:
    predictions = resp.json()
    print("Predictions received:")
    print(json.dumps(predictions, indent=2))
else:
    print(f"Error ({resp.status_code}): {resp.text}")
    print("\nThe endpoint may still be warming up. Try again in a minute.")

{
  "dataframe_records": [
    {
      "date": "2024-07-01T00:00:00.000",
      "product_id": "SKU-001"
    }
  ]
}